# VoxoL — Parakeet FR/EN autonomous training gate

Choose a GPU runtime, then use **Runtime → Run all**. The only interaction is Google's Drive
authorization. The notebook selects a safe T4/L4/A100 profile, resumes verified downloads,
trains an architecture-compatible candidate, and compares the
official source model with the candidate on locked FLEURS FR/EN and MediaSpeech FR tests.

The candidate is never promoted merely because training completed. It must preserve both FLEURS
languages, improve MediaSpeech by at least 10% relative, and reduce empty transcripts. Expect
several hours and roughly 12 GB of free Google Drive space. The trainer stores only the best
fine-tuned parameter delta; the evaluator reconstructs the official model from that delta.


In [ ]:
from pathlib import Path
import json
import os
import shutil

from google.colab import drive
import torch

drive.mount("/content/drive")
if not torch.cuda.is_available():
    raise RuntimeError("Select Runtime → Change runtime type → GPU, then run all again.")

gpu = torch.cuda.get_device_properties(0)
memory_gib = gpu.total_memory / (1024 ** 3)
bf16 = torch.cuda.is_bf16_supported()
if memory_gib < 14:
    raise RuntimeError(f"The assigned GPU has only {memory_gib:.1f} GiB; at least 14 GiB is required.")

if memory_gib < 20:
    PROFILE = {
        "name": "t4-safe",
        "precision": "16-mixed",
        "batch_size": 1,
        "validation_batch_size": 1,
        "accumulate_grad_batches": 16,
        "max_duration": 12,
        "train_top_encoder_layers": 6,
        "evaluation_batch_size": 4,
    }
elif memory_gib < 38:
    PROFILE = {
        "name": "l4-safe",
        "precision": "bf16-mixed" if bf16 else "16-mixed",
        "batch_size": 2,
        "validation_batch_size": 2,
        "accumulate_grad_batches": 8,
        "max_duration": 18,
        "train_top_encoder_layers": 8,
        "evaluation_batch_size": 8,
    }
else:
    PROFILE = {
        "name": "a100-safe",
        "precision": "bf16-mixed" if bf16 else "16-mixed",
        "batch_size": 4,
        "validation_batch_size": 4,
        "accumulate_grad_batches": 4,
        "max_duration": 30,
        "train_top_encoder_layers": 8,
        "evaluation_batch_size": 16,
    }

DRIVE_ROOT = Path("/content/drive/MyDrive/VoxoL-Parakeet")
SCRATCH_ROOT = Path("/content/voxol")
NEMO_ROOT = Path("/content/NeMo")
for directory in (DRIVE_ROOT, SCRATCH_ROOT):
    directory.mkdir(parents=True, exist_ok=True)

os.environ["HF_HOME"] = str(SCRATCH_ROOT / "huggingface")
os.environ["TOKENIZERS_PARALLELISM"] = "false"
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"
torch.set_float32_matmul_precision("high")

run_profile = {
    **PROFILE,
    "gpu": gpu.name,
    "memoryGiB": round(memory_gib, 2),
    "bf16Supported": bf16,
    "scratchFreeGiB": round(shutil.disk_usage("/content").free / (1024 ** 3), 2),
}
(DRIVE_ROOT / "run-profile.json").write_text(
    json.dumps(run_profile, indent=2, sort_keys=True) + "\n",
    encoding="utf-8",
)
print(json.dumps(run_profile, indent=2, sort_keys=True))


In [ ]:
import importlib
import importlib.util
import subprocess
import sys

NEMO_REVISION = "2381f42f6979449b5b99538f8f80135831009b51"

def checked(command):
    print("+", " ".join(map(str, command)), flush=True)
    subprocess.run(list(map(str, command)), check=True)

checked(["apt-get", "-qq", "update"])
checked(["apt-get", "-qq", "install", "-y", "libsndfile1", "ffmpeg"])
if not (NEMO_ROOT / ".git").exists():
    NEMO_ROOT.mkdir(parents=True, exist_ok=True)
    checked(["git", "-C", NEMO_ROOT, "init"])
    checked(["git", "-C", NEMO_ROOT, "remote", "add", "origin", "https://github.com/NVIDIA-NeMo/NeMo.git"])
checked(["git", "-C", NEMO_ROOT, "fetch", "--depth=1", "origin", NEMO_REVISION])
checked(["git", "-C", NEMO_ROOT, "checkout", "--detach", "FETCH_HEAD"])
actual_revision = subprocess.check_output(
    ["git", "-C", NEMO_ROOT, "rev-parse", "HEAD"],
    text=True,
).strip()
if actual_revision != NEMO_REVISION:
    raise RuntimeError(f"NeMo revision mismatch: {actual_revision}")

checked([sys.executable, "-m", "pip", "install", "-q", "--upgrade", "pip"])
checked([sys.executable, "-m", "pip", "install", "-q", "-e", f"{NEMO_ROOT}[asr]"])

nemo_package = NEMO_ROOT / "nemo" / "__init__.py"
if not nemo_package.is_file():
    raise RuntimeError(f"NeMo checkout does not contain the Python package: {nemo_package}")
if str(NEMO_ROOT) not in sys.path:
    sys.path.insert(0, str(NEMO_ROOT))
importlib.invalidate_caches()
if importlib.util.find_spec("nemo") is None:
    package_report = subprocess.run(
        [sys.executable, "-m", "pip", "show", "nemo_toolkit"],
        capture_output=True,
        text=True,
        check=False,
    )
    raise RuntimeError(
        "NeMo was installed but is not importable in the current Colab kernel.\n"
        f"pip show output:\n{package_report.stdout or package_report.stderr}"
    )
import nemo
print(f"NeMo ready at {actual_revision}")


In [ ]:
import json
from pathlib import Path

EMBEDDED_SOURCES = json.loads(r'''{
  "Scripts/prepare-fleurs-test-benchmark.py": "#!/usr/bin/env python3\n\"\"\"Prepare the complete pinned FLEURS French and English test splits.\"\"\"\n\nfrom __future__ import annotations\n\nimport argparse\nimport csv\nimport hashlib\nimport json\nimport os\nfrom pathlib import Path\nimport shutil\nimport tarfile\n\nimport resumable_dataset_download\n\n\nDATASET_REVISION = \"70bb2e84b976b7e960aa89f1c648e09c59f894dd\"\nCONFIGURATIONS = {\n    \"en_us\": {\n        \"language\": \"english\",\n        \"archive_sha256\": (\n            \"d9c2e37b41aacd41bc283554a0a82b5476b36887049774ecb2819dcaaa55a356\"\n        ),\n        \"archive_bytes\": 289_851_356,\n        \"tsv_sha256\": (\n            \"74c046239374deeb60fa63f258f907388093a32bcaa3140965f70ef05c79f7ca\"\n        ),\n    },\n    \"fr_fr\": {\n        \"language\": \"french\",\n        \"archive_sha256\": (\n            \"d23690e102f373554d1b544cd2ff1e76e4fedeb04953c0b72751a1b7c518cfdd\"\n        ),\n        \"archive_bytes\": 349_036_055,\n        \"tsv_sha256\": (\n            \"5d06d338b242e00786fcf12c4c92008b9f399d5a5c872c91dca90572e7869c0d\"\n        ),\n    },\n}\n\n\ndef source_url(locale: str, relative_path: str) -> str:\n    return (\n        \"https://huggingface.co/datasets/google/fleurs/resolve/\"\n        f\"{DATASET_REVISION}/data/{locale}/{relative_path}\"\n    )\n\n\ndef download(\n    locale: str,\n    relative_path: str,\n    expected_sha256: str,\n    destination: Path,\n    expected_bytes: int | None = None,\n) -> Path:\n    return resumable_dataset_download.download_verified(\n        source_url(locale, relative_path),\n        expected_sha256,\n        destination,\n        expected_bytes,\n    )\n\n\ndef load_rows(source: Path) -> list[dict[str, str]]:\n    rows = []\n    with source.open(encoding=\"utf-8\", newline=\"\") as stream:\n        for line_number, columns in enumerate(\n            csv.reader(stream, delimiter=\"\\t\", quoting=csv.QUOTE_NONE),\n            1,\n        ):\n            if len(columns) != 7:\n                raise SystemExit(f\"Unexpected FLEURS row: {source}:{line_number}\")\n            rows.append(\n                {\n                    \"sentence_id\": columns[0],\n                    \"audio_name\": columns[1],\n                    \"clean\": columns[2].strip(),\n                    \"verbatim\": columns[3].strip(),\n                    \"gender\": columns[6].lower(),\n                }\n            )\n    return rows\n\n\ndef extract_items(\n    archive_path: Path,\n    rows: list[dict[str, str]],\n    locale: str,\n    output_root: Path,\n) -> list[dict[str, object]]:\n    configuration = CONFIGURATIONS[locale]\n    rows_by_name = {row[\"audio_name\"]: row for row in rows}\n    extracted = set()\n    audio_root = output_root / \"audio\"\n    items = []\n\n    with tarfile.open(archive_path, mode=\"r:gz\") as archive:\n        for member in archive:\n            audio_name = Path(member.name).name\n            row = rows_by_name.get(audio_name)\n            if row is None or not member.isfile():\n                continue\n            source = archive.extractfile(member)\n            if source is None:\n                raise SystemExit(f\"Could not read FLEURS audio: {audio_name}\")\n            identity = hashlib.sha256(\n                f\"{locale}\\0{row['sentence_id']}\\0{audio_name}\".encode()\n            ).hexdigest()[:12]\n            identifier = f\"fleurs-{locale}-test-{identity}\"\n            relative_audio_path = Path(f\"fleurs-{locale}-test\") / f\"{identifier}.wav\"\n            destination = audio_root / relative_audio_path\n            if not destination.exists() or destination.stat().st_size != member.size:\n                destination.parent.mkdir(parents=True, exist_ok=True)\n                temporary = destination.with_suffix(\".wav.partial\")\n                with temporary.open(\"wb\") as output:\n                    shutil.copyfileobj(source, output)\n                if temporary.stat().st_size != member.size:\n                    temporary.unlink(missing_ok=True)\n                    raise SystemExit(f\"Incomplete FLEURS audio: {audio_name}\")\n                os.replace(temporary, destination)\n            extracted.add(audio_name)\n            items.append(\n                {\n                    \"id\": identifier,\n                    \"audioPath\": relative_audio_path.as_posix(),\n                    \"speakerID\": f\"fleurs-{locale}-test-speaker-unknown\",\n                    \"sessionID\": f\"fleurs-{locale}-test\",\n                    \"split\": \"blind\",\n                    \"language\": configuration[\"language\"],\n                    \"microphone\": \"fleurs-source\",\n                    \"environment\": \"source-unknown\",\n                    \"tags\": [\n                        \"public\",\n                        \"fleurs\",\n                        \"official-test\",\n                        \"read-speech\",\n                        row[\"gender\"],\n                    ],\n                    \"reference\": {\n                        \"verbatim\": row[\"verbatim\"],\n                        \"clean\": row[\"clean\"],\n                        \"criticalSpans\": [],\n                        \"reviewed\": True,\n                    },\n                }\n            )\n\n    missing = rows_by_name.keys() - extracted\n    if missing:\n        raise SystemExit(f\"FLEURS {locale} archive is missing {len(missing)} files.\")\n    return items\n\n\ndef main() -> None:\n    parser = argparse.ArgumentParser()\n    parser.add_argument(\"--cache-root\", type=Path, required=True)\n    parser.add_argument(\"--output-root\", type=Path, required=True)\n    parser.add_argument(\n        \"--locale\",\n        action=\"append\",\n        choices=sorted(CONFIGURATIONS),\n        help=\"Locale to prepare. The default prepares both pinned locales.\",\n    )\n    arguments = parser.parse_args()\n    locales = arguments.locale or sorted(CONFIGURATIONS)\n    items = []\n\n    for locale in locales:\n        configuration = CONFIGURATIONS[locale]\n        tsv = download(\n            locale,\n            \"test.tsv\",\n            str(configuration[\"tsv_sha256\"]),\n            arguments.cache_root / f\"{locale}-test.tsv\",\n        )\n        archive = download(\n            locale,\n            \"audio/test.tar.gz\",\n            str(configuration[\"archive_sha256\"]),\n            arguments.cache_root / f\"{locale}-test.tar.gz\",\n            int(configuration[\"archive_bytes\"]),\n        )\n        items.extend(\n            extract_items(\n                archive,\n                load_rows(tsv),\n                locale,\n                arguments.output_root,\n            )\n        )\n\n    items.sort(key=lambda item: str(item[\"id\"]))\n    manifest = {\n        \"schemaVersion\": 1,\n        \"benchmarkID\": (\n            \"voxol-fleurs-test-\" f\"{'-'.join(locales)}-{DATASET_REVISION[:12]}\"\n        ),\n        \"normalizationVersion\": \"voxol-asr-normalizer-v1\",\n        \"items\": items,\n    }\n    arguments.output_root.mkdir(parents=True, exist_ok=True)\n    manifest_path = arguments.output_root / \"manifest-unfrozen.json\"\n    manifest_path.write_text(\n        json.dumps(manifest, ensure_ascii=False, indent=2, sort_keys=True) + \"\\n\",\n        encoding=\"utf-8\",\n    )\n    print(manifest_path)\n\n\nif __name__ == \"__main__\":\n    main()\n",
  "Scripts/prepare-mediaspeech-fr-benchmark.py": "#!/usr/bin/env python3\n\"\"\"Prepare the complete official OpenSLR MediaSpeech French corpus.\"\"\"\n\nfrom __future__ import annotations\n\nimport argparse\nimport json\nimport os\nfrom pathlib import Path\nimport shutil\nimport tarfile\n\nimport resumable_dataset_download\n\n\nARCHIVE_URL = \"https://www.openslr.org/resources/108/FR.tgz\"\nARCHIVE_SHA256 = \"edefa83dab25acc2c99d18605a9362e0d7d28953435f128efaabf3bbda79f390\"\n\n\ndef download(destination: Path) -> Path:\n    return resumable_dataset_download.download_verified(\n        ARCHIVE_URL,\n        ARCHIVE_SHA256,\n        destination,\n    )\n\n\ndef build_manifest(archive_path: Path, output_root: Path) -> Path:\n    audio_root = output_root / \"audio\"\n    transcriptions = {}\n    audio_identifiers = set()\n    with tarfile.open(archive_path, mode=\"r:gz\") as archive:\n        for member in archive:\n            if not member.isfile():\n                continue\n            source = archive.extractfile(member)\n            if source is None:\n                raise SystemExit(f\"Could not read MediaSpeech item: {member.name}\")\n            source_identifier = Path(member.name).stem\n            if member.name.endswith(\".txt\"):\n                transcription = source.read().decode(\"utf-8\").strip()\n                if not transcription:\n                    raise SystemExit(f\"Empty MediaSpeech transcript: {member.name}\")\n                transcriptions[source_identifier] = transcription\n                continue\n            if not member.name.endswith(\".flac\"):\n                continue\n            audio_identifiers.add(source_identifier)\n            relative_audio_path = Path(\"mediaspeech-fr\") / f\"{source_identifier}.flac\"\n            destination = audio_root / relative_audio_path\n            if not destination.exists() or destination.stat().st_size != member.size:\n                destination.parent.mkdir(parents=True, exist_ok=True)\n                temporary = destination.with_suffix(\".flac.partial\")\n                with temporary.open(\"wb\") as output:\n                    shutil.copyfileobj(source, output)\n                if temporary.stat().st_size != member.size:\n                    temporary.unlink(missing_ok=True)\n                    raise SystemExit(f\"Incomplete MediaSpeech audio: {member.name}\")\n                os.replace(temporary, destination)\n\n    missing_transcripts = audio_identifiers - transcriptions.keys()\n    missing_audio = transcriptions.keys() - audio_identifiers\n    if missing_transcripts or missing_audio:\n        raise SystemExit(\"MediaSpeech archive has unpaired audio or transcript items.\")\n    items = []\n    for source_identifier in sorted(audio_identifiers):\n        relative_audio_path = Path(\"mediaspeech-fr\") / f\"{source_identifier}.flac\"\n        transcription = transcriptions[source_identifier]\n        items.append(\n            {\n                \"id\": f\"mediaspeech-fr-{source_identifier}\",\n                \"audioPath\": relative_audio_path.as_posix(),\n                \"speakerID\": \"mediaspeech-fr-speaker-unknown\",\n                \"sessionID\": \"mediaspeech-fr-media-unknown\",\n                \"split\": \"blind\",\n                \"language\": \"french\",\n                \"microphone\": \"media-source\",\n                \"environment\": \"real-media\",\n                \"tags\": [\n                    \"public\",\n                    \"mediaspeech\",\n                    \"official-benchmark\",\n                    \"media\",\n                ],\n                \"reference\": {\n                    \"verbatim\": transcription,\n                    \"clean\": transcription,\n                    \"criticalSpans\": [],\n                    \"reviewed\": True,\n                },\n            }\n        )\n\n    manifest = {\n        \"schemaVersion\": 1,\n        \"benchmarkID\": \"voxol-mediaspeech-fr-openslr108\",\n        \"normalizationVersion\": \"voxol-asr-normalizer-v1\",\n        \"items\": items,\n    }\n    output_root.mkdir(parents=True, exist_ok=True)\n    manifest_path = output_root / \"manifest-unfrozen.json\"\n    manifest_path.write_text(\n        json.dumps(manifest, ensure_ascii=False, indent=2, sort_keys=True) + \"\\n\",\n        encoding=\"utf-8\",\n    )\n    return manifest_path\n\n\ndef main() -> None:\n    parser = argparse.ArgumentParser()\n    parser.add_argument(\"--cache-root\", type=Path, required=True)\n    parser.add_argument(\"--output-root\", type=Path, required=True)\n    arguments = parser.parse_args()\n    archive = download(arguments.cache_root / \"FR.tgz\")\n    print(build_manifest(archive, arguments.output_root))\n\n\nif __name__ == \"__main__\":\n    main()\n",
  "Scripts/prepare-parakeet-fleurs-finetune.py": "#!/usr/bin/env python3\n\"\"\"Prepare pinned FLEURS French/English NeMo manifests for Parakeet fine-tuning.\"\"\"\n\nfrom __future__ import annotations\n\nimport argparse\nimport csv\nimport json\nimport os\nfrom pathlib import Path\nimport shutil\nimport tarfile\n\nimport resumable_dataset_download\n\n\nDATASET_REVISION = \"70bb2e84b976b7e960aa89f1c648e09c59f894dd\"\nSAMPLE_RATE = 16_000\nCONFIGURATIONS = {\n    \"en_us\": {\n        \"train_archive_sha256\": (\n            \"5f4491948c2bd29ac00f4b8afae2378f0a1dcdde4041b5cd284a80dff01fa9f5\"\n        ),\n        \"train_archive_bytes\": 1_380_572_241,\n        \"train_tsv_sha256\": (\n            \"3ccfc83672cc03a835143e325abb38b4163e3a21725bc1a7d1165bc309b95852\"\n        ),\n        \"dev_archive_sha256\": (\n            \"2658fda72f199e12676ecac9415094667a4e14e149b146e568ea00b2a2f0954c\"\n        ),\n        \"dev_tsv_sha256\": (\n            \"9d57ee7e91e9d4c92edb39f6bbea668ef8dc2a3ff96eb510d5580b2ad05d17ec\"\n        ),\n    },\n    \"fr_fr\": {\n        \"train_archive_sha256\": (\n            \"39b5bad1f61d3ae4ef64c2eb16f9524ffb21d8568965dd71b09702befdbd7f95\"\n        ),\n        \"train_archive_bytes\": 1_730_529_959,\n        \"train_tsv_sha256\": (\n            \"3df668c3b9b4101cc3e3d8f3024b311ccfd2e9a8c8f910b3efedc27fd4219a0f\"\n        ),\n        \"dev_archive_sha256\": (\n            \"f2f065dec3b02212e27151c51162d2213df55d0a8efc6b88e36992673ddf66e6\"\n        ),\n        \"dev_tsv_sha256\": (\n            \"3e0b792358c1cb4a426fe1c18fc1571d5406b390d738a2ed1bfd3c8b9d28de44\"\n        ),\n    },\n}\n\n\ndigest = resumable_dataset_download.digest\n\n\ndef source_url(locale: str, relative_path: str) -> str:\n    return (\n        \"https://huggingface.co/datasets/google/fleurs/resolve/\"\n        f\"{DATASET_REVISION}/data/{locale}/{relative_path}\"\n    )\n\n\ndef download(\n    locale: str,\n    relative_path: str,\n    expected_sha256: str,\n    destination: Path,\n    expected_bytes: int | None = None,\n) -> Path:\n    return resumable_dataset_download.download_verified(\n        source_url(locale, relative_path),\n        expected_sha256,\n        destination,\n        expected_bytes,\n    )\n\n\ndef load_rows(source: Path) -> list[dict[str, object]]:\n    rows = []\n    with source.open(encoding=\"utf-8\", newline=\"\") as stream:\n        for line_number, columns in enumerate(\n            csv.reader(stream, delimiter=\"\\t\", quoting=csv.QUOTE_NONE),\n            1,\n        ):\n            if len(columns) != 7:\n                raise SystemExit(f\"Unexpected FLEURS row: {source}:{line_number}\")\n            try:\n                sample_count = int(columns[5])\n            except ValueError as error:\n                raise SystemExit(\n                    f\"Invalid FLEURS sample count: {source}:{line_number}\"\n                ) from error\n            text = \" \".join(columns[2].strip().split())\n            if not text or sample_count < 1:\n                raise SystemExit(f\"Invalid FLEURS item: {source}:{line_number}\")\n            rows.append(\n                {\n                    \"sentence_id\": columns[0],\n                    \"audio_name\": columns[1],\n                    \"text\": text,\n                    \"sample_count\": sample_count,\n                }\n            )\n    return rows\n\n\ndef extract_audio(\n    archive_path: Path,\n    rows: list[dict[str, object]],\n    locale: str,\n    split: str,\n    output_root: Path,\n) -> None:\n    rows_by_name = {str(row[\"audio_name\"]): row for row in rows}\n    extracted = set()\n    destination_root = output_root / \"audio\" / locale / split\n    destination_root.mkdir(parents=True, exist_ok=True)\n\n    with tarfile.open(archive_path, mode=\"r:gz\") as archive:\n        for member in archive:\n            audio_name = Path(member.name).name\n            if audio_name not in rows_by_name or not member.isfile():\n                continue\n            source = archive.extractfile(member)\n            if source is None:\n                raise SystemExit(f\"Could not read FLEURS audio: {audio_name}\")\n            destination = destination_root / audio_name\n            if not destination.exists() or destination.stat().st_size != member.size:\n                temporary = destination.with_suffix(\".wav.partial\")\n                with temporary.open(\"wb\") as output:\n                    shutil.copyfileobj(source, output)\n                if temporary.stat().st_size != member.size:\n                    temporary.unlink(missing_ok=True)\n                    raise SystemExit(f\"Incomplete FLEURS audio: {audio_name}\")\n                os.replace(temporary, destination)\n            extracted.add(audio_name)\n\n    missing = rows_by_name.keys() - extracted\n    if missing:\n        raise SystemExit(\n            f\"FLEURS {locale}/{split} archive is missing {len(missing)} files.\"\n        )\n\n\ndef nemo_record(\n    row: dict[str, object],\n    locale: str,\n    split: str,\n    output_root: Path,\n) -> dict[str, object]:\n    audio_path = (\n        output_root / \"audio\" / locale / split / str(row[\"audio_name\"])\n    ).resolve()\n    return {\n        \"audio_filepath\": str(audio_path),\n        \"duration\": round(int(row[\"sample_count\"]) / SAMPLE_RATE, 6),\n        \"text\": str(row[\"text\"]),\n    }\n\n\ndef interleave(records: dict[str, list[dict[str, object]]]) -> list[dict[str, object]]:\n    combined = []\n    for index in range(max(map(len, records.values()))):\n        for locale in sorted(records):\n            if index < len(records[locale]):\n                combined.append(records[locale][index])\n    return combined\n\n\ndef write_jsonl(path: Path, records: list[dict[str, object]]) -> None:\n    temporary = path.with_suffix(path.suffix + \".partial\")\n    with temporary.open(\"w\", encoding=\"utf-8\") as stream:\n        for record in records:\n            stream.write(json.dumps(record, ensure_ascii=False, sort_keys=True) + \"\\n\")\n    os.replace(temporary, path)\n\n\ndef plan(\n    cache_root: Path,\n    output_root: Path,\n    splits: tuple[str, ...] = (\"train\", \"dev\"),\n) -> dict[str, object]:\n    return {\n        \"dataset\": \"google/fleurs\",\n        \"revision\": DATASET_REVISION,\n        \"locales\": sorted(CONFIGURATIONS),\n        \"requestedSplits\": list(splits),\n        \"trainingSplits\": [\"train\"] if \"train\" in splits else [],\n        \"validationSplits\": [\"dev\"] if \"dev\" in splits else [],\n        \"forbiddenEvaluationSplits\": [\"test\", \"MediaSpeech\"],\n        \"downloadBytesKnownMinimum\": sum(\n            int(configuration[\"train_archive_bytes\"])\n            for configuration in CONFIGURATIONS.values()\n        ) if \"train\" in splits else 0,\n        \"cacheRoot\": str(cache_root.resolve()),\n        \"outputRoot\": str(output_root.resolve()),\n    }\n\n\ndef main() -> None:\n    parser = argparse.ArgumentParser()\n    parser.add_argument(\"--cache-root\", type=Path, required=True)\n    parser.add_argument(\"--output-root\", type=Path, required=True)\n    parser.add_argument(\n        \"--print-plan\",\n        action=\"store_true\",\n        help=\"Print the pinned inputs without downloading them.\",\n    )\n    parser.add_argument(\n        \"--split\",\n        action=\"append\",\n        choices=(\"train\", \"dev\"),\n        help=\"Split to prepare. The default prepares train and dev.\",\n    )\n    arguments = parser.parse_args()\n    splits = tuple(dict.fromkeys(arguments.split or (\"train\", \"dev\")))\n\n    if arguments.print_plan:\n        print(\n            json.dumps(\n                plan(arguments.cache_root, arguments.output_root, splits),\n                indent=2,\n            )\n        )\n        return\n\n    records_by_split: dict[str, dict[str, list[dict[str, object]]]] = {\n        split: {} for split in splits\n    }\n    provenance: dict[str, object] = plan(\n        arguments.cache_root,\n        arguments.output_root,\n        splits,\n    )\n    provenance[\"sources\"] = {}\n    provenance[\"counts\"] = {}\n    provenance[\"durationHours\"] = {}\n\n    for locale, configuration in sorted(CONFIGURATIONS.items()):\n        provenance[\"sources\"][locale] = {}\n        provenance[\"counts\"][locale] = {}\n        provenance[\"durationHours\"][locale] = {}\n        for split in splits:\n            tsv = download(\n                locale,\n                f\"{split}.tsv\",\n                str(configuration[f\"{split}_tsv_sha256\"]),\n                arguments.cache_root / f\"{locale}-{split}.tsv\",\n            )\n            archive = download(\n                locale,\n                f\"audio/{split}.tar.gz\",\n                str(configuration[f\"{split}_archive_sha256\"]),\n                arguments.cache_root / f\"{locale}-{split}.tar.gz\",\n                int(configuration[\"train_archive_bytes\"]) if split == \"train\" else None,\n            )\n            rows = load_rows(tsv)\n            extract_audio(\n                archive,\n                rows,\n                locale,\n                split,\n                arguments.output_root,\n            )\n            records = [\n                nemo_record(row, locale, split, arguments.output_root) for row in rows\n            ]\n            records_by_split[split][locale] = records\n            provenance[\"sources\"][locale][split] = {\n                \"archiveSHA256\": digest(archive),\n                \"tsvSHA256\": digest(tsv),\n            }\n            provenance[\"counts\"][locale][split] = len(records)\n            provenance[\"durationHours\"][locale][split] = round(\n                sum(float(record[\"duration\"]) for record in records) / 3600,\n                4,\n            )\n\n    arguments.output_root.mkdir(parents=True, exist_ok=True)\n    if \"train\" in records_by_split:\n        write_jsonl(\n            arguments.output_root / \"train.jsonl\",\n            interleave(records_by_split[\"train\"]),\n        )\n    if \"dev\" in records_by_split:\n        write_jsonl(\n            arguments.output_root / \"validation.jsonl\",\n            interleave(records_by_split[\"dev\"]),\n        )\n    provenance_path = arguments.output_root / \"provenance.json\"\n    provenance_path.write_text(\n        json.dumps(provenance, ensure_ascii=False, indent=2, sort_keys=True) + \"\\n\",\n        encoding=\"utf-8\",\n    )\n    print(provenance_path)\n\n\nif __name__ == \"__main__\":\n    main()\n",
  "Scripts/resumable_dataset_download.py": "#!/usr/bin/env python3\n\"\"\"Verified, resumable downloads with automatic generated-cache recovery.\"\"\"\n\nfrom __future__ import annotations\n\nimport hashlib\nimport os\nfrom pathlib import Path\nimport shutil\nimport subprocess\nimport sys\n\n\ndef digest(path: Path) -> str:\n    hasher = hashlib.sha256()\n    with path.open(\"rb\") as source:\n        while chunk := source.read(1024 * 1024):\n            hasher.update(chunk)\n    return hasher.hexdigest()\n\n\ndef validation_error(\n    path: Path,\n    expected_sha256: str,\n    expected_bytes: int | None,\n) -> str | None:\n    if not path.is_file():\n        return \"file is missing\"\n    actual_bytes = path.stat().st_size\n    if expected_bytes is not None and actual_bytes != expected_bytes:\n        return f\"size is {actual_bytes} bytes; expected {expected_bytes}\"\n    actual_sha256 = digest(path)\n    if actual_sha256 != expected_sha256:\n        return f\"SHA-256 is {actual_sha256}; expected {expected_sha256}\"\n    return None\n\n\ndef remove_invalid_generated_file(path: Path, reason: str) -> None:\n    print(\n        f\"[dataset-cache] Removing invalid generated file {path}: {reason}\",\n        file=sys.stderr,\n        flush=True,\n    )\n    path.unlink(missing_ok=True)\n\n\ndef download_verified(\n    url: str,\n    expected_sha256: str,\n    destination: Path,\n    expected_bytes: int | None = None,\n) -> Path:\n    if destination.exists():\n        reason = validation_error(destination, expected_sha256, expected_bytes)\n        if reason is None:\n            print(f\"[dataset-cache] Verified {destination}\", flush=True)\n            return destination\n        remove_invalid_generated_file(destination, reason)\n\n    destination.parent.mkdir(parents=True, exist_ok=True)\n    partial = destination.with_suffix(destination.suffix + \".partial\")\n    curl = shutil.which(\"curl\")\n    if curl is None:\n        raise SystemExit(\"curl is required to download the datasets.\")\n\n    for attempt in range(1, 3):\n        if (\n            partial.exists()\n            and expected_bytes is not None\n            and partial.stat().st_size > expected_bytes\n        ):\n            remove_invalid_generated_file(\n                partial,\n                f\"size exceeds the expected {expected_bytes} bytes\",\n            )\n        print(\n            f\"[dataset-cache] Downloading {destination.name} \"\n            f\"(attempt {attempt}/2, resume enabled)\",\n            flush=True,\n        )\n        try:\n            subprocess.run(\n                [\n                    curl,\n                    \"--fail\",\n                    \"--location\",\n                    \"--show-error\",\n                    \"--retry\",\n                    \"5\",\n                    \"--retry-delay\",\n                    \"2\",\n                    \"--retry-all-errors\",\n                    \"--connect-timeout\",\n                    \"30\",\n                    \"--continue-at\",\n                    \"-\",\n                    \"--output\",\n                    str(partial),\n                    url,\n                ],\n                check=True,\n            )\n        except subprocess.CalledProcessError as error:\n            raise SystemExit(\n                f\"Dataset download failed (curl exit {error.returncode}): {url}\\n\"\n                f\"The partial file was kept for resume: {partial}\\n\"\n                \"Check the Colab network and free Drive space, then rerun this cell.\"\n            ) from error\n\n        reason = validation_error(partial, expected_sha256, expected_bytes)\n        if reason is None:\n            os.replace(partial, destination)\n            print(f\"[dataset-cache] Ready: {destination}\", flush=True)\n            return destination\n        remove_invalid_generated_file(partial, reason)\n        if attempt == 1:\n            print(\n                \"[dataset-cache] Retrying once from byte zero.\",\n                file=sys.stderr,\n                flush=True,\n            )\n\n    raise SystemExit(\n        f\"Dataset verification failed twice: {url}\\n\"\n        \"The invalid partial file was removed. Rerun the cell or inspect Drive health.\"\n    )\n",
  "Tools/training/freeze_asr_manifest.py": "#!/usr/bin/env python3\n\"\"\"Freeze a VoxoL ASR manifest using the Swift manifest digest contract.\"\"\"\n\nfrom __future__ import annotations\n\nimport argparse\nfrom datetime import datetime, timezone\nimport hashlib\nimport json\nfrom pathlib import Path\n\n\ndef canonical_bytes(manifest: dict[str, object]) -> bytes:\n    canonical = {\n        \"benchmarkID\": manifest[\"benchmarkID\"],\n        \"frozenAt\": manifest[\"frozenAt\"],\n        \"items\": manifest[\"items\"],\n        \"normalizationVersion\": manifest[\"normalizationVersion\"],\n        \"schemaVersion\": manifest[\"schemaVersion\"],\n    }\n    return json.dumps(\n        canonical,\n        ensure_ascii=False,\n        separators=(\",\", \":\"),\n        sort_keys=True,\n    ).encode()\n\n\ndef main() -> None:\n    parser = argparse.ArgumentParser()\n    parser.add_argument(\"--input\", type=Path, required=True)\n    parser.add_argument(\"--output\", type=Path, required=True)\n    parser.add_argument(\n        \"--timestamp\",\n        help=\"Explicit ISO-8601 freeze timestamp for a reproducible manifest.\",\n    )\n    arguments = parser.parse_args()\n    manifest = json.loads(arguments.input.read_text(encoding=\"utf-8\"))\n    if manifest.get(\"contentSHA256\") or manifest.get(\"frozenAt\"):\n        raise SystemExit(\"Input manifest is already frozen.\")\n    manifest[\"frozenAt\"] = arguments.timestamp or (\n        datetime.now(timezone.utc).isoformat().replace(\"+00:00\", \"Z\")\n    )\n    manifest[\"contentSHA256\"] = hashlib.sha256(canonical_bytes(manifest)).hexdigest()\n    arguments.output.parent.mkdir(parents=True, exist_ok=True)\n    arguments.output.write_text(\n        json.dumps(manifest, ensure_ascii=False, indent=2, sort_keys=True) + \"\\n\",\n        encoding=\"utf-8\",\n    )\n    print(arguments.output)\n\n\nif __name__ == \"__main__\":\n    main()\n",
  "Tools/training/run_nemo_asr_benchmark.py": "#!/usr/bin/env python3\n\"\"\"Run a NeMo checkpoint or a VoxoL trainable delta on a frozen benchmark.\"\"\"\n\nfrom __future__ import annotations\n\nimport argparse\nimport hashlib\nimport json\nfrom pathlib import Path\nimport time\n\n\nMODEL_ID = \"nvidia/parakeet-tdt-0.6b-v3\"\nMODEL_REVISION = \"7c35754d166cca382ad1e53e68b01e7c575f3a1d\"\nMODEL_FILENAME = \"parakeet-tdt-0.6b-v3.nemo\"\n\n\ndef sha256(path: Path) -> str:\n    digest = hashlib.sha256()\n    with path.open(\"rb\") as stream:\n        for chunk in iter(lambda: stream.read(8 * 1024 * 1024), b\"\"):\n            digest.update(chunk)\n    return digest.hexdigest()\n\n\ndef read_jsonl(path: Path) -> list[dict[str, object]]:\n    if not path.exists():\n        return []\n    source = path.read_text(encoding=\"utf-8\")\n    lines = source.splitlines()\n    rows = []\n    for line_number, line in enumerate(lines, 1):\n        if not line.strip():\n            continue\n        try:\n            rows.append(json.loads(line))\n        except json.JSONDecodeError:\n            is_trailing_partial = line_number == len(lines) and not source.endswith(\n                \"\\n\"\n            )\n            if not is_trailing_partial:\n                raise\n            recovered = \"\\n\".join(lines[:-1])\n            path.write_text(\n                recovered + (\"\\n\" if recovered else \"\"),\n                encoding=\"utf-8\",\n            )\n            print(\n                f\"Removed an incomplete trailing prediction from {path}.\",\n                flush=True,\n            )\n    return rows\n\n\ndef text_of(result: object) -> str:\n    if isinstance(result, str):\n        return result.strip()\n    text = getattr(result, \"text\", None)\n    if isinstance(text, str):\n        return text.strip()\n    raise TypeError(f\"Unsupported NeMo transcription result: {type(result).__name__}\")\n\n\ndef apply_trainable_delta(\n    model: object,\n    delta_path: Path,\n    torch: object,\n    *,\n    base_artifact_sha256: str | None = None,\n) -> dict[str, object]:\n    payload = torch.load(delta_path, map_location=\"cpu\", weights_only=True)\n    if not isinstance(payload, dict) or payload.get(\"schemaVersion\") not in (1, 2):\n        raise SystemExit(\"Unsupported VoxoL trainable delta.\")\n    if payload.get(\"baseModel\") != MODEL_ID:\n        raise SystemExit(\n            f\"Delta base model mismatch: expected {MODEL_ID}, \"\n            f\"got {payload.get('baseModel')}.\"\n        )\n    schema_version = int(payload[\"schemaVersion\"])\n    state_key = \"stateDict\" if schema_version == 1 else \"stateDelta\"\n    state_dict = payload.get(state_key)\n    if not isinstance(state_dict, dict) or not state_dict:\n        raise SystemExit(\"The VoxoL trainable delta contains no tensors.\")\n    if schema_version == 2:\n        if payload.get(\"artifactType\") != \"voxol-parameter-delta\":\n            raise SystemExit(\"Unsupported VoxoL parameter-delta type.\")\n        if payload.get(\"baseRevision\") != MODEL_REVISION:\n            raise SystemExit(\"The VoxoL delta uses a different base revision.\")\n        expected_base_digest = str(payload.get(\"baseArtifactSHA256\", \"\"))\n        if (\n            base_artifact_sha256 is None\n            or expected_base_digest != base_artifact_sha256\n        ):\n            raise SystemExit(\"The VoxoL delta uses a different base artifact.\")\n    model_state = model.state_dict()\n    unexpected = sorted(set(state_dict) - set(model_state))\n    if unexpected:\n        raise SystemExit(f\"Delta contains an unknown tensor: {unexpected[0]}\")\n    with torch.no_grad():\n        for name, source in state_dict.items():\n            destination = model_state[name]\n            if tuple(source.shape) != tuple(destination.shape):\n                raise SystemExit(f\"Delta tensor shape mismatch: {name}\")\n            update = source.to(\n                device=destination.device,\n                dtype=destination.dtype,\n            )\n            if schema_version == 1:\n                destination.copy_(update)\n            else:\n                destination.add_(update)\n    return payload\n\n\ndef load_pinned_model(\n    nemo_asr: object,\n    map_location: str,\n) -> tuple[object, str]:\n    from huggingface_hub import hf_hub_download\n\n    artifact = Path(\n        hf_hub_download(\n            repo_id=MODEL_ID,\n            filename=MODEL_FILENAME,\n            revision=MODEL_REVISION,\n        )\n    )\n    model = nemo_asr.models.ASRModel.restore_from(\n        restore_path=str(artifact),\n        map_location=map_location,\n    )\n    return model, sha256(artifact)\n\n\ndef main() -> None:\n    parser = argparse.ArgumentParser()\n    model_group = parser.add_mutually_exclusive_group(required=True)\n    model_group.add_argument(\"--model\", type=Path)\n    model_group.add_argument(\"--delta\", type=Path)\n    model_group.add_argument(\"--pretrained-name\")\n    parser.add_argument(\"--manifest\", type=Path, required=True)\n    parser.add_argument(\"--audio-root\", type=Path, required=True)\n    parser.add_argument(\"--output\", type=Path, required=True)\n    parser.add_argument(\"--batch-size\", type=int, default=8)\n    parser.add_argument(\"--resume\", action=\"store_true\")\n    arguments = parser.parse_args()\n    if arguments.batch_size < 1:\n        raise SystemExit(\"--batch-size must be positive.\")\n\n    import torch\n    import nemo.collections.asr as nemo_asr\n\n    if not torch.cuda.is_available():\n        raise SystemExit(\"This evaluator requires an NVIDIA CUDA GPU.\")\n    manifest = json.loads(arguments.manifest.read_text(encoding=\"utf-8\"))\n    if not manifest.get(\"contentSHA256\") or not manifest.get(\"frozenAt\"):\n        raise SystemExit(\"The VoxoL benchmark manifest must be frozen.\")\n    items = list(manifest[\"items\"])\n    existing = read_jsonl(arguments.output) if arguments.resume else []\n    completed = {str(row[\"id\"]) for row in existing}\n    pending = [item for item in items if str(item[\"id\"]) not in completed]\n\n    if arguments.delta is not None:\n        model, base_digest = load_pinned_model(\n            nemo_asr,\n            \"cuda\",\n        )\n        delta_metadata = apply_trainable_delta(\n            model,\n            arguments.delta,\n            torch,\n            base_artifact_sha256=base_digest,\n        )\n        model_identity = (\n            f\"{MODEL_ID}+{arguments.delta}\" f\"@epoch-{delta_metadata['epoch']}\"\n        )\n    elif arguments.model is not None:\n        model = nemo_asr.models.ASRModel.restore_from(\n            restore_path=str(arguments.model),\n            map_location=\"cuda\",\n        )\n        model_identity = str(arguments.model)\n    else:\n        if str(arguments.pretrained_name) == MODEL_ID:\n            model, base_digest = load_pinned_model(nemo_asr, \"cuda\")\n            model_identity = f\"{MODEL_ID}@{MODEL_REVISION}:{base_digest[:12]}\"\n        else:\n            model = nemo_asr.models.ASRModel.from_pretrained(\n                model_name=str(arguments.pretrained_name),\n                map_location=\"cuda\",\n            )\n            model_identity = str(arguments.pretrained_name)\n    model = model.cuda().eval()\n    arguments.output.parent.mkdir(parents=True, exist_ok=True)\n    mode = \"a\" if arguments.resume else \"w\"\n    with arguments.output.open(mode, encoding=\"utf-8\") as output:\n        for offset in range(0, len(pending), arguments.batch_size):\n            batch = pending[offset : offset + arguments.batch_size]\n            paths = [\n                str((arguments.audio_root / str(item[\"audioPath\"])).resolve())\n                for item in batch\n            ]\n            missing = [path for path in paths if not Path(path).is_file()]\n            if missing:\n                raise SystemExit(f\"Missing benchmark audio: {missing[0]}\")\n            started = time.perf_counter()\n            with torch.inference_mode():\n                results = model.transcribe(\n                    audio=paths,\n                    batch_size=len(batch),\n                    verbose=False,\n                )\n            elapsed = time.perf_counter() - started\n            for item, result in zip(batch, results, strict=True):\n                transcript = text_of(result)\n                row = {\n                    \"id\": item[\"id\"],\n                    \"rawText\": transcript,\n                    \"finalText\": transcript,\n                    \"checkpoint\": model_identity,\n                    \"inferenceMilliseconds\": elapsed * 1_000 / len(batch),\n                }\n                output.write(json.dumps(row, ensure_ascii=False, sort_keys=True) + \"\\n\")\n                output.flush()\n            completed_count = min(offset + len(batch), len(pending))\n            print(\n                f\"[{completed_count}/{len(pending)}] \"\n                f\"{elapsed * 1_000 / len(batch):.1f} ms/item\",\n                flush=True,\n            )\n\n\nif __name__ == \"__main__\":\n    main()\n",
  "Tools/training/run_voxol_nemo_finetune.py": "#!/usr/bin/env python3\n\"\"\"Fine-tune Parakeet with a memory-bounded, architecture-preserving recipe.\"\"\"\n\nfrom __future__ import annotations\n\nimport argparse\nfrom dataclasses import asdict, dataclass\nimport gc\nimport hashlib\nimport json\nimport os\nfrom pathlib import Path\nimport re\nfrom typing import Iterable\n\ntry:\n    from Tools.training.score_asr_predictions import edit_distance, normalize\nexcept ModuleNotFoundError:\n    from score_asr_predictions import edit_distance, normalize\n\n\nMODEL_ID = \"nvidia/parakeet-tdt-0.6b-v3\"\nMODEL_REVISION = \"7c35754d166cca382ad1e53e68b01e7c575f3a1d\"\nMODEL_FILENAME = \"parakeet-tdt-0.6b-v3.nemo\"\nENCODER_LAYER_PATTERN = re.compile(r\"^encoder\\.layers\\.(\\d+)\\.\")\n\n\n@dataclass(frozen=True)\nclass TrainingConfiguration:\n    train_manifest: str\n    validation_manifest: str\n    experiment_root: str\n    precision: str\n    batch_size: int\n    validation_batch_size: int\n    accumulate_grad_batches: int\n    max_duration: float\n    train_top_encoder_layers: int\n    train_decoder: bool\n    train_joint: bool\n    freeze_batchnorm: bool\n    max_epochs: int\n    max_steps: int\n    learning_rate: float\n    minimum_learning_rate: float\n    warmup_steps: int\n    checkpoint_every_n_steps: int\n    num_workers: int\n    seed: int\n    deterministic: bool\n    resume_checkpoint: str | None\n\n\n@dataclass(frozen=True)\nclass ValidationItem:\n    audio_path: str\n    reference: str\n\n\ndef parser() -> argparse.ArgumentParser:\n    result = argparse.ArgumentParser()\n    result.add_argument(\"--train-manifest\", type=Path, required=True)\n    result.add_argument(\"--validation-manifest\", type=Path, required=True)\n    result.add_argument(\"--experiment-root\", type=Path, required=True)\n    result.add_argument(\"--precision\", choices=(\"16-mixed\", \"bf16-mixed\"), required=True)\n    result.add_argument(\"--batch-size\", type=int, required=True)\n    result.add_argument(\"--validation-batch-size\", type=int, required=True)\n    result.add_argument(\"--accumulate-grad-batches\", type=int, required=True)\n    result.add_argument(\"--max-duration\", type=float, required=True)\n    result.add_argument(\"--train-top-encoder-layers\", type=int, default=8)\n    result.add_argument(\"--freeze-decoder\", action=\"store_true\")\n    result.add_argument(\"--freeze-joint\", action=\"store_true\")\n    result.add_argument(\"--freeze-batchnorm\", action=\"store_true\")\n    result.add_argument(\"--max-epochs\", type=int, default=5)\n    result.add_argument(\"--max-steps\", type=int, default=0)\n    result.add_argument(\"--learning-rate\", type=float, default=2e-5)\n    result.add_argument(\"--minimum-learning-rate\", type=float, default=2e-6)\n    result.add_argument(\"--warmup-steps\", type=int, default=100)\n    result.add_argument(\"--checkpoint-every-n-steps\", type=int, default=0)\n    result.add_argument(\"--num-workers\", type=int, default=2)\n    result.add_argument(\"--seed\", type=int, default=42)\n    result.add_argument(\"--deterministic\", action=\"store_true\")\n    result.add_argument(\"--resume-checkpoint\", type=Path)\n    result.add_argument(\"--dry-run\", action=\"store_true\")\n    return result\n\n\ndef validated_configuration(arguments: argparse.Namespace) -> TrainingConfiguration:\n    positive_values = {\n        \"batch size\": arguments.batch_size,\n        \"validation batch size\": arguments.validation_batch_size,\n        \"gradient accumulation\": arguments.accumulate_grad_batches,\n        \"maximum duration\": arguments.max_duration,\n        \"trained encoder layers\": arguments.train_top_encoder_layers,\n        \"epochs\": arguments.max_epochs,\n        \"learning rate\": arguments.learning_rate,\n        \"minimum learning rate\": arguments.minimum_learning_rate,\n        \"workers\": arguments.num_workers,\n    }\n    invalid = [name for name, value in positive_values.items() if value <= 0]\n    if invalid:\n        raise SystemExit(f\"Values must be positive: {', '.join(invalid)}\")\n    nonnegative_values = {\n        \"maximum steps\": arguments.max_steps,\n        \"warmup steps\": arguments.warmup_steps,\n        \"checkpoint interval\": arguments.checkpoint_every_n_steps,\n    }\n    invalid_nonnegative = [\n        name for name, value in nonnegative_values.items() if value < 0\n    ]\n    if invalid_nonnegative:\n        raise SystemExit(\n            f\"Values must be nonnegative: {', '.join(invalid_nonnegative)}\"\n        )\n    for manifest in (arguments.train_manifest, arguments.validation_manifest):\n        if not arguments.dry_run and (not manifest.is_file() or manifest.stat().st_size == 0):\n            raise SystemExit(f\"Missing training manifest: {manifest}\")\n    if (\n        arguments.resume_checkpoint is not None\n        and not arguments.dry_run\n        and (\n            not arguments.resume_checkpoint.is_file()\n            or arguments.resume_checkpoint.stat().st_size == 0\n        )\n    ):\n        raise SystemExit(\n            f\"Missing resume checkpoint: {arguments.resume_checkpoint}\"\n        )\n    return TrainingConfiguration(\n        train_manifest=str(arguments.train_manifest.resolve()),\n        validation_manifest=str(arguments.validation_manifest.resolve()),\n        experiment_root=str(arguments.experiment_root.resolve()),\n        precision=arguments.precision,\n        batch_size=arguments.batch_size,\n        validation_batch_size=arguments.validation_batch_size,\n        accumulate_grad_batches=arguments.accumulate_grad_batches,\n        max_duration=arguments.max_duration,\n        train_top_encoder_layers=arguments.train_top_encoder_layers,\n        train_decoder=not arguments.freeze_decoder,\n        train_joint=not arguments.freeze_joint,\n        freeze_batchnorm=arguments.freeze_batchnorm,\n        max_epochs=arguments.max_epochs,\n        max_steps=arguments.max_steps,\n        learning_rate=arguments.learning_rate,\n        minimum_learning_rate=arguments.minimum_learning_rate,\n        warmup_steps=arguments.warmup_steps,\n        checkpoint_every_n_steps=arguments.checkpoint_every_n_steps,\n        num_workers=arguments.num_workers,\n        seed=arguments.seed,\n        deterministic=arguments.deterministic,\n        resume_checkpoint=(\n            str(arguments.resume_checkpoint.resolve())\n            if arguments.resume_checkpoint is not None\n            else None\n        ),\n    )\n\n\ndef encoder_layer_indices(names: Iterable[str]) -> list[int]:\n    return sorted(\n        {\n            int(match.group(1))\n            for name in names\n            if (match := ENCODER_LAYER_PATTERN.match(name)) is not None\n        }\n    )\n\n\ndef parameter_should_train(\n    name: str,\n    first_trainable_encoder_layer: int,\n    *,\n    train_decoder: bool = True,\n    train_joint: bool = True,\n    batchnorm_names: frozenset[str] = frozenset(),\n) -> bool:\n    if name in batchnorm_names:\n        return False\n    match = ENCODER_LAYER_PATTERN.match(name)\n    if match is not None:\n        return int(match.group(1)) >= first_trainable_encoder_layer\n    if train_decoder and name.startswith(\"decoder.\"):\n        return True\n    return train_joint and name.startswith(\"joint.\")\n\n\ndef batchnorm_state_names(model: object, torch: object) -> frozenset[str]:\n    names = set()\n    batchnorm_base = torch.nn.modules.batchnorm._BatchNorm\n    for module_name, module in model.named_modules():\n        if not isinstance(module, batchnorm_base):\n            continue\n        prefix = f\"{module_name}.\" if module_name else \"\"\n        names.update(prefix + name for name, _ in module.named_parameters(recurse=False))\n        names.update(prefix + name for name, _ in module.named_buffers(recurse=False))\n    return frozenset(names)\n\n\ndef freeze_batchnorm_modules(model: object, torch: object) -> None:\n    batchnorm_base = torch.nn.modules.batchnorm._BatchNorm\n    for module in model.modules():\n        if isinstance(module, batchnorm_base):\n            module.eval()\n\n\ndef configure_trainable_parameters(\n    model: object,\n    top_layer_count: int,\n    *,\n    train_decoder: bool = True,\n    train_joint: bool = True,\n    freeze_batchnorm: bool = False,\n    torch: object | None = None,\n) -> tuple[int, int]:\n    named_parameters = list(model.named_parameters())\n    indices = encoder_layer_indices(name for name, _ in named_parameters)\n    if not indices or indices != list(range(indices[-1] + 1)):\n        raise RuntimeError(\"Unexpected Parakeet encoder layer names.\")\n    if top_layer_count > len(indices):\n        raise RuntimeError(\n            f\"Requested {top_layer_count} trainable encoder layers, model has {len(indices)}.\"\n        )\n    first_trainable = len(indices) - top_layer_count\n    if freeze_batchnorm and torch is None:\n        raise RuntimeError(\"torch is required when BatchNorm is frozen.\")\n    batchnorm_names = (\n        batchnorm_state_names(model, torch) if freeze_batchnorm else frozenset()\n    )\n    trainable = 0\n    total = 0\n    for name, parameter in named_parameters:\n        parameter.requires_grad = parameter_should_train(\n            name,\n            first_trainable,\n            train_decoder=train_decoder,\n            train_joint=train_joint,\n            batchnorm_names=batchnorm_names,\n        )\n        parameter_count = parameter.numel()\n        total += parameter_count\n        if parameter.requires_grad:\n            trainable += parameter_count\n    if trainable == 0:\n        raise RuntimeError(\"The selective fine-tuning recipe selected no parameters.\")\n    return trainable, total\n\n\ndef delta_checkpoint_path(experiment_root: Path) -> Path:\n    return experiment_root / \"best-trainable-parameters.delta.pt\"\n\n\ndef atomic_trainer_checkpoint(\n    trainer: object,\n    destination: Path,\n) -> None:\n    destination.parent.mkdir(parents=True, exist_ok=True)\n    temporary = destination.with_suffix(destination.suffix + \".partial\")\n    temporary.unlink(missing_ok=True)\n    trainer.save_checkpoint(temporary, weights_only=False)\n    os.replace(temporary, destination)\n\n\ndef experiment_manager_configuration(experiment_root: str) -> dict[str, object]:\n    return {\n        \"exp_dir\": experiment_root,\n        \"name\": \"voxol-parakeet-finetune\",\n        \"create_tensorboard_logger\": True,\n        \"create_checkpoint_callback\": False,\n        \"resume_if_exists\": False,\n        \"resume_ignore_no_checkpoint\": True,\n        \"resume_past_end\": False,\n    }\n\n\ndef capture_trainable_base_state(\n    model: object,\n    torch: object,\n) -> dict[str, object]:\n    selected = {\n        name: parameter.detach()\n        .to(device=\"cpu\", dtype=torch.float32)\n        .contiguous()\n        .clone()\n        for name, parameter in model.named_parameters()\n        if parameter.requires_grad\n    }\n    if not selected:\n        raise RuntimeError(\"The selective fine-tuning recipe selected no parameters.\")\n    return selected\n\n\ndef true_parameter_delta(\n    model: object,\n    base_state: dict[str, object],\n    torch: object,\n) -> dict[str, object]:\n    current = dict(model.named_parameters())\n    missing = sorted(set(base_state) - set(current))\n    if missing:\n        raise RuntimeError(f\"The model lost a trainable tensor: {missing[0]}\")\n    return {\n        name: (\n            current[name]\n            .detach()\n            .to(device=\"cpu\", dtype=torch.float32)\n            .contiguous()\n            - base\n        )\n        for name, base in base_state.items()\n    }\n\n\ndef sha256(path: Path) -> str:\n    digest = hashlib.sha256()\n    with path.open(\"rb\") as stream:\n        for chunk in iter(lambda: stream.read(8 * 1024 * 1024), b\"\"):\n            digest.update(chunk)\n    return digest.hexdigest()\n\n\ndef load_validation_items(manifest_path: Path) -> list[ValidationItem]:\n    items = []\n    for line_number, line in enumerate(\n        manifest_path.read_text(encoding=\"utf-8\").splitlines(),\n        1,\n    ):\n        if not line.strip():\n            continue\n        row = json.loads(line)\n        audio_value = row.get(\"audio_filepath\", row.get(\"audio_path\"))\n        reference = str(row.get(\"text\", \"\")).strip()\n        if not audio_value or not reference:\n            raise RuntimeError(\n                f\"Invalid validation row at {manifest_path}:{line_number}\"\n            )\n        audio_path = Path(str(audio_value))\n        if not audio_path.is_absolute():\n            audio_path = manifest_path.parent / audio_path\n        items.append(\n            ValidationItem(\n                audio_path=str(audio_path.resolve()),\n                reference=reference,\n            )\n        )\n    if not items:\n        raise RuntimeError(f\"Empty validation manifest: {manifest_path}\")\n    return items\n\n\ndef transcription_text(result: object) -> str:\n    if isinstance(result, str):\n        return result.strip()\n    text = getattr(result, \"text\", None)\n    if isinstance(text, str):\n        return text.strip()\n    raise TypeError(f\"Unsupported NeMo transcription result: {type(result).__name__}\")\n\n\ndef normalized_validation_score(\n    references: list[str],\n    hypotheses: list[str],\n) -> dict[str, float | int | str]:\n    if len(references) != len(hypotheses):\n        raise RuntimeError(\n            \"Validation transcription count does not match the manifest.\"\n        )\n    word_errors = 0\n    reference_words = 0\n    for reference, hypothesis in zip(references, hypotheses, strict=True):\n        reference_tokens = normalize(reference).split()\n        hypothesis_tokens = normalize(hypothesis).split()\n        word_errors += edit_distance(reference_tokens, hypothesis_tokens)\n        reference_words += len(reference_tokens)\n    if reference_words == 0:\n        raise RuntimeError(\"Validation manifest contains no reference words.\")\n    return {\n        \"metric\": \"voxol-asr-v1-micro-wer\",\n        \"microWER\": word_errors / reference_words,\n        \"referenceWords\": reference_words,\n        \"wordErrors\": word_errors,\n    }\n\n\ndef score_validation_model(\n    model: object,\n    items: list[ValidationItem],\n    batch_size: int,\n) -> dict[str, float | int | str]:\n    references = []\n    hypotheses = []\n    for offset in range(0, len(items), batch_size):\n        batch = items[offset : offset + batch_size]\n        results = model.transcribe(\n            audio=[item.audio_path for item in batch],\n            batch_size=len(batch),\n            verbose=False,\n        )\n        references.extend(item.reference for item in batch)\n        hypotheses.extend(transcription_text(result) for result in results)\n    return normalized_validation_score(references, hypotheses)\n\n\ndef main() -> None:\n    arguments = parser().parse_args()\n    configuration = validated_configuration(arguments)\n    if arguments.dry_run:\n        print(json.dumps(asdict(configuration), indent=2, sort_keys=True))\n        return\n\n    import lightning.pytorch as pl\n    from huggingface_hub import hf_hub_download\n    from nemo.collections.asr.models import ASRModel\n    from nemo.utils.exp_manager import exp_manager\n    from omegaconf import OmegaConf\n    import torch\n\n    if not torch.cuda.is_available():\n        raise SystemExit(\"A CUDA-capable NVIDIA GPU is required.\")\n    if (\n        configuration.precision == \"bf16-mixed\"\n        and not torch.cuda.is_bf16_supported()\n    ):\n        raise SystemExit(\"The selected GPU does not support BF16.\")\n\n    pl.seed_everything(configuration.seed, workers=True)\n\n    class KeepFrozenModulesInEvaluationMode(pl.Callback):\n        def __init__(\n            self,\n            trained_top_layer_count: int,\n            freeze_batchnorm: bool,\n        ) -> None:\n            self.trained_top_layer_count = trained_top_layer_count\n            self.freeze_batchnorm = freeze_batchnorm\n\n        def enforce(self, pl_module: object) -> None:\n            layers = list(pl_module.encoder.layers)\n            frozen_count = len(layers) - self.trained_top_layer_count\n            if frozen_count < 0:\n                raise RuntimeError(\"The model has fewer encoder layers than requested.\")\n            for layer in layers[:frozen_count]:\n                layer.eval()\n            if self.freeze_batchnorm:\n                freeze_batchnorm_modules(pl_module, torch)\n\n        def on_train_epoch_start(self, trainer: object, pl_module: object) -> None:\n            del trainer\n            self.enforce(pl_module)\n\n        def on_train_batch_start(\n            self,\n            trainer: object,\n            pl_module: object,\n            batch: object,\n            batch_index: int,\n        ) -> None:\n            del trainer, batch, batch_index\n            self.enforce(pl_module)\n\n    experiment_root = Path(configuration.experiment_root)\n    experiment_root.mkdir(parents=True, exist_ok=True)\n    delta_path = delta_checkpoint_path(experiment_root)\n    full_checkpoint_base = os.environ.get(\"VOXOL_FULL_CHECKPOINT_ROOT\")\n    full_checkpoint_root = (\n        Path(full_checkpoint_base).resolve() / experiment_root.name\n        if full_checkpoint_base\n        else experiment_root / \"checkpoints\"\n    )\n    base_artifact = Path(\n        hf_hub_download(\n            repo_id=MODEL_ID,\n            filename=MODEL_FILENAME,\n            revision=MODEL_REVISION,\n        )\n    )\n    base_artifact_sha256 = sha256(base_artifact)\n\n    model = ASRModel.restore_from(\n        restore_path=str(base_artifact),\n        map_location=\"cpu\",\n    )\n    for metric in (\n        getattr(model, \"wer\", None),\n        getattr(getattr(model, \"joint\", None), \"_wer\", None),\n    ):\n        if metric is not None and hasattr(metric, \"log_prediction\"):\n            metric.log_prediction = False\n    trainable, total = configure_trainable_parameters(\n        model,\n        configuration.train_top_encoder_layers,\n        train_decoder=configuration.train_decoder,\n        train_joint=configuration.train_joint,\n        freeze_batchnorm=configuration.freeze_batchnorm,\n        torch=torch,\n    )\n    trainable_base_state = capture_trainable_base_state(model, torch)\n    validation_items = load_validation_items(\n        Path(configuration.validation_manifest)\n    )\n\n    class SaveBestTrainableDelta(pl.Callback):\n        def __init__(\n            self,\n            output_path: Path,\n            items: list[ValidationItem],\n            batch_size: int,\n        ) -> None:\n            self.output_path = output_path\n            self.items = items\n            self.batch_size = batch_size\n            self.best_selection_wer = float(\"inf\")\n            if output_path.is_file() and output_path.stat().st_size > 0:\n                existing = torch.load(\n                    output_path,\n                    map_location=\"cpu\",\n                    weights_only=True,\n                )\n                self.best_selection_wer = float(\n                    existing[\"validationWERSelection\"]\n                )\n            self.saved_steps: set[int] = set()\n\n        def payload(\n            self,\n            trainer: object,\n            pl_module: object,\n            raw_nemo_wer: float,\n            selection_score: dict[str, float | int | str],\n        ) -> dict[str, object]:\n            return {\n                \"schemaVersion\": 2,\n                \"artifactType\": \"voxol-parameter-delta\",\n                \"baseModel\": MODEL_ID,\n                \"baseRevision\": MODEL_REVISION,\n                \"baseArtifactSHA256\": base_artifact_sha256,\n                \"epoch\": int(trainer.current_epoch),\n                \"globalStep\": int(trainer.global_step),\n                \"validationWERInternal\": raw_nemo_wer,\n                \"validationWERSelection\": selection_score[\"microWER\"],\n                \"validationWERSelectionMetric\": selection_score[\"metric\"],\n                \"validationReferenceWords\": selection_score[\"referenceWords\"],\n                \"validationWordErrors\": selection_score[\"wordErrors\"],\n                \"trainedTopEncoderLayers\": (\n                    configuration.train_top_encoder_layers\n                ),\n                \"trainDecoder\": configuration.train_decoder,\n                \"trainJoint\": configuration.train_joint,\n                \"batchNormFrozen\": configuration.freeze_batchnorm,\n                \"stateDelta\": true_parameter_delta(\n                    pl_module,\n                    trainable_base_state,\n                    torch,\n                ),\n            }\n\n        def save_payload(\n            self,\n            path: Path,\n            payload: dict[str, object],\n        ) -> None:\n            path.parent.mkdir(parents=True, exist_ok=True)\n            temporary_path = path.with_suffix(path.suffix + \".partial\")\n            torch.save(payload, temporary_path)\n            temporary_path.replace(path)\n\n        def on_validation_end(self, trainer: object, pl_module: object) -> None:\n            if trainer.sanity_checking:\n                return\n            step = int(trainer.global_step)\n            if step in self.saved_steps:\n                return\n            metric = trainer.callback_metrics.get(\"val_wer\")\n            if metric is None:\n                raise RuntimeError(\"Validation completed without val_wer.\")\n            raw_nemo_wer = float(metric.detach().float().cpu())\n            selection_score = score_validation_model(\n                pl_module,\n                self.items,\n                self.batch_size,\n            )\n            selection_wer = float(selection_score[\"microWER\"])\n            payload = self.payload(\n                trainer,\n                pl_module,\n                raw_nemo_wer,\n                selection_score,\n            )\n            checkpoint_root = experiment_root / \"checkpoints\"\n            self.save_payload(\n                checkpoint_root / f\"step-{step:06d}.delta.pt\",\n                payload,\n            )\n            if configuration.checkpoint_every_n_steps > 0:\n                atomic_trainer_checkpoint(\n                    trainer,\n                    full_checkpoint_root / f\"step-{step:06d}.ckpt\",\n                )\n            self.saved_steps.add(step)\n            if selection_wer < self.best_selection_wer:\n                self.save_payload(self.output_path, payload)\n                self.best_selection_wer = selection_wer\n                print(\n                    \"Saved trainable FP32 parameter delta at \"\n                    f\"VoxoL normalized val_wer={selection_wer:.5f} \"\n                    f\"(NeMo callback val_wer={raw_nemo_wer:.5f}): \"\n                    f\"{self.output_path}\",\n                    flush=True,\n                )\n            del payload\n            gc.collect()\n\n    validation_interval_batches = (\n        configuration.checkpoint_every_n_steps\n        * configuration.accumulate_grad_batches\n        if configuration.checkpoint_every_n_steps > 0\n        else None\n    )\n    trainer = pl.Trainer(\n        accelerator=\"gpu\",\n        devices=1,\n        strategy=\"auto\",\n        precision=configuration.precision,\n        max_epochs=configuration.max_epochs,\n        max_steps=configuration.max_steps or -1,\n        accumulate_grad_batches=configuration.accumulate_grad_batches,\n        gradient_clip_val=1.0,\n        sync_batchnorm=False,\n        num_sanity_val_steps=0,\n        check_val_every_n_epoch=(\n            None if validation_interval_batches is not None else 1\n        ),\n        val_check_interval=validation_interval_batches,\n        log_every_n_steps=10,\n        enable_checkpointing=False,\n        logger=False,\n        benchmark=False,\n        deterministic=configuration.deterministic,\n        callbacks=[\n            KeepFrozenModulesInEvaluationMode(\n                configuration.train_top_encoder_layers,\n                configuration.freeze_batchnorm,\n            ),\n            SaveBestTrainableDelta(\n                delta_path,\n                validation_items,\n                configuration.validation_batch_size,\n            ),\n        ],\n    )\n    model.set_trainer(trainer)\n    exp_manager(\n        trainer,\n        OmegaConf.create(\n            experiment_manager_configuration(str(experiment_root))\n        ),\n    )\n\n    summary = {\n        **asdict(configuration),\n        \"model\": MODEL_ID,\n        \"modelRevision\": MODEL_REVISION,\n        \"baseArtifact\": str(base_artifact),\n        \"baseArtifactSHA256\": base_artifact_sha256,\n        \"cudaDevice\": torch.cuda.get_device_name(0),\n        \"trainableParameters\": trainable,\n        \"totalParameters\": total,\n        \"trainableFraction\": trainable / total,\n        \"fullCheckpointRoot\": str(full_checkpoint_root),\n    }\n    (experiment_root / \"training-configuration.json\").write_text(\n        json.dumps(summary, indent=2, sort_keys=True) + \"\\n\",\n        encoding=\"utf-8\",\n    )\n    print(json.dumps(summary, indent=2, sort_keys=True), flush=True)\n\n    model.setup_training_data(\n        OmegaConf.create(\n            {\n                \"manifest_filepath\": configuration.train_manifest,\n                \"sample_rate\": 16_000,\n                \"batch_size\": configuration.batch_size,\n                \"shuffle\": True,\n                \"num_workers\": configuration.num_workers,\n                \"pin_memory\": True,\n                \"max_duration\": configuration.max_duration,\n                \"min_duration\": 0.1,\n                \"is_tarred\": False,\n                \"tarred_audio_filepaths\": None,\n                \"shuffle_n\": 2_048,\n                \"bucketing_strategy\": \"fully_randomized\",\n                \"bucketing_batch_size\": None,\n            }\n        )\n    )\n    model.setup_multiple_validation_data(\n        OmegaConf.create(\n            {\n                \"manifest_filepath\": configuration.validation_manifest,\n                \"sample_rate\": 16_000,\n                \"batch_size\": configuration.validation_batch_size,\n                \"shuffle\": False,\n                \"use_start_end_token\": False,\n                \"num_workers\": configuration.num_workers,\n                \"pin_memory\": True,\n            }\n        )\n    )\n    model.setup_optimization(\n        OmegaConf.create(\n            {\n                \"name\": \"adamw\",\n                \"lr\": configuration.learning_rate,\n                \"betas\": [0.9, 0.98],\n                \"weight_decay\": 1e-3,\n                \"sched\": {\n                    \"name\": \"CosineAnnealing\",\n                    \"warmup_steps\": configuration.warmup_steps,\n                    \"warmup_ratio\": None,\n                    \"min_lr\": configuration.minimum_learning_rate,\n                },\n            }\n        )\n    )\n    model.spec_augment = ASRModel.from_config_dict(\n        OmegaConf.create(\n            {\n                \"_target_\": \"nemo.collections.asr.modules.SpectrogramAugmentation\",\n                \"freq_masks\": 2,\n                \"time_masks\": 10,\n                \"freq_width\": 27,\n                \"time_width\": 0.05,\n            }\n        )\n    )\n    trainer.fit(model, ckpt_path=configuration.resume_checkpoint)\n    if not delta_path.is_file() or delta_path.stat().st_size == 0:\n        raise RuntimeError(\"Training completed without a trainable delta checkpoint.\")\n\n\nif __name__ == \"__main__\":\n    main()\n",
  "Tools/training/score_asr_predictions.py": "#!/usr/bin/env python3\n\"\"\"Score NeMo predictions with VoxoL's public ASR normalization contract.\"\"\"\n\nfrom __future__ import annotations\n\nimport argparse\nfrom collections.abc import Sequence\nimport hashlib\nimport json\nfrom pathlib import Path\nimport statistics\nimport unicodedata\n\n\ndef normalize(text: str) -> str:\n    canonical = unicodedata.normalize(\"NFC\", text).replace(\"’\", \"'\").replace(\"‘\", \"'\")\n    canonical = canonical.lower()\n    output = []\n    previous_was_space = True\n    for character in canonical:\n        keep = character.isalpha() or character.isdecimal() or character == \"'\"\n        if keep:\n            output.append(character)\n            previous_was_space = False\n        elif not previous_was_space:\n            output.append(\" \")\n            previous_was_space = True\n    return \"\".join(output).strip()\n\n\ndef edit_distance(reference: Sequence[object], hypothesis: Sequence[object]) -> int:\n    previous = list(range(len(hypothesis) + 1))\n    for reference_index, reference_item in enumerate(reference, 1):\n        current = [reference_index]\n        for hypothesis_index, hypothesis_item in enumerate(hypothesis, 1):\n            if reference_item == hypothesis_item:\n                current.append(previous[hypothesis_index - 1])\n            else:\n                current.append(\n                    1\n                    + min(\n                        previous[hypothesis_index - 1],\n                        previous[hypothesis_index],\n                        current[hypothesis_index - 1],\n                    )\n                )\n        previous = current\n    return previous[-1]\n\n\ndef edit_operation_counts(\n    reference: Sequence[object],\n    hypothesis: Sequence[object],\n) -> tuple[int, int, int]:\n    distances = [\n        [0] * (len(hypothesis) + 1)\n        for _ in range(len(reference) + 1)\n    ]\n    for reference_index in range(len(reference) + 1):\n        distances[reference_index][0] = reference_index\n    for hypothesis_index in range(len(hypothesis) + 1):\n        distances[0][hypothesis_index] = hypothesis_index\n    for reference_index, reference_item in enumerate(reference, 1):\n        for hypothesis_index, hypothesis_item in enumerate(hypothesis, 1):\n            substitution_cost = 0 if reference_item == hypothesis_item else 1\n            distances[reference_index][hypothesis_index] = min(\n                distances[reference_index - 1][hypothesis_index - 1]\n                + substitution_cost,\n                distances[reference_index - 1][hypothesis_index] + 1,\n                distances[reference_index][hypothesis_index - 1] + 1,\n            )\n\n    substitutions = 0\n    deletions = 0\n    insertions = 0\n    reference_index = len(reference)\n    hypothesis_index = len(hypothesis)\n    while reference_index or hypothesis_index:\n        if (\n            reference_index\n            and hypothesis_index\n            and reference[reference_index - 1] == hypothesis[hypothesis_index - 1]\n            and distances[reference_index][hypothesis_index]\n            == distances[reference_index - 1][hypothesis_index - 1]\n        ):\n            reference_index -= 1\n            hypothesis_index -= 1\n        elif (\n            reference_index\n            and hypothesis_index\n            and distances[reference_index][hypothesis_index]\n            == distances[reference_index - 1][hypothesis_index - 1] + 1\n        ):\n            substitutions += 1\n            reference_index -= 1\n            hypothesis_index -= 1\n        elif (\n            reference_index\n            and distances[reference_index][hypothesis_index]\n            == distances[reference_index - 1][hypothesis_index] + 1\n        ):\n            deletions += 1\n            reference_index -= 1\n        else:\n            insertions += 1\n            hypothesis_index -= 1\n    return substitutions, deletions, insertions\n\n\ndef score_items(\n    items: list[dict[str, object]],\n    predictions: dict[str, dict[str, object]],\n) -> dict[str, object]:\n    macro_wer = 0.0\n    word_errors = 0\n    reference_words = 0\n    character_errors = 0\n    reference_characters = 0\n    exact_matches = 0\n    empty_outputs = 0\n    substitutions = 0\n    deletions = 0\n    insertions = 0\n    by_language: dict[str, list[tuple[int, int]]] = {}\n    operations_by_language: dict[str, list[int]] = {}\n    latencies = []\n\n    for item in items:\n        identifier = str(item[\"id\"])\n        if identifier not in predictions:\n            raise SystemExit(f\"Missing prediction: {identifier}\")\n        reference = normalize(str(item[\"reference\"][\"verbatim\"]))\n        hypothesis = normalize(str(predictions[identifier][\"rawText\"]))\n        reference_word_list = reference.split()\n        hypothesis_word_list = hypothesis.split()\n        errors = edit_distance(reference_word_list, hypothesis_word_list)\n        item_substitutions, item_deletions, item_insertions = edit_operation_counts(\n            reference_word_list,\n            hypothesis_word_list,\n        )\n        if item_substitutions + item_deletions + item_insertions != errors:\n            raise RuntimeError(f\"Inconsistent edit alignment for {identifier}\")\n        word_count = len(reference_word_list)\n        macro_wer += errors / word_count if word_count else 0.0\n        word_errors += errors\n        reference_words += word_count\n        character_errors += edit_distance(list(reference), list(hypothesis))\n        reference_characters += len(reference)\n        exact_matches += reference == hypothesis\n        empty_outputs += not hypothesis\n        substitutions += item_substitutions\n        deletions += item_deletions\n        insertions += item_insertions\n        language = str(item[\"language\"])\n        by_language.setdefault(language, []).append((errors, word_count))\n        language_operations = operations_by_language.setdefault(\n            language,\n            [0, 0, 0],\n        )\n        language_operations[0] += item_substitutions\n        language_operations[1] += item_deletions\n        language_operations[2] += item_insertions\n        latency = predictions[identifier].get(\"inferenceMilliseconds\")\n        if isinstance(latency, (int, float)) and latency >= 0:\n            latencies.append(float(latency))\n\n    count = len(items)\n    sorted_latencies = sorted(latencies)\n\n    def percentile(fraction: float) -> float | None:\n        if not sorted_latencies:\n            return None\n        position = fraction * (len(sorted_latencies) - 1)\n        lower = int(position)\n        upper = min(lower + 1, len(sorted_latencies) - 1)\n        weight = position - lower\n        return (\n            sorted_latencies[lower] * (1 - weight)\n            + sorted_latencies[upper] * weight\n        )\n\n    return {\n        \"itemCount\": count,\n        \"macroWER\": macro_wer / count if count else 0.0,\n        \"microWER\": word_errors / reference_words if reference_words else 0.0,\n        \"microCER\": (\n            character_errors / reference_characters if reference_characters else 0.0\n        ),\n        \"exactMatchRate\": exact_matches / count if count else 0.0,\n        \"emptyOutputCount\": empty_outputs,\n        \"wordErrors\": {\n            \"substitutions\": substitutions,\n            \"deletions\": deletions,\n            \"insertions\": insertions,\n            \"referenceWords\": reference_words,\n            \"deletionRate\": deletions / reference_words if reference_words else 0.0,\n        },\n        \"latencyMilliseconds\": {\n            \"sampleCount\": len(sorted_latencies),\n            \"mean\": statistics.fmean(sorted_latencies) if sorted_latencies else None,\n            \"p50\": percentile(0.50),\n            \"p95\": percentile(0.95),\n            \"p99\": percentile(0.99),\n        },\n        \"byLanguage\": {\n            language: {\n                \"itemCount\": len(values),\n                \"microWER\": (\n                    sum(errors for errors, _ in values)\n                    / sum(words for _, words in values)\n                ),\n                \"wordErrors\": {\n                    \"substitutions\": operations_by_language[language][0],\n                    \"deletions\": operations_by_language[language][1],\n                    \"insertions\": operations_by_language[language][2],\n                    \"referenceWords\": sum(words for _, words in values),\n                    \"deletionRate\": (\n                        operations_by_language[language][1]\n                        / sum(words for _, words in values)\n                    ),\n                },\n            }\n            for language, values in sorted(by_language.items())\n        },\n    }\n\n\ndef load_predictions(path: Path) -> dict[str, dict[str, object]]:\n    predictions = {}\n    for line_number, line in enumerate(path.read_text(encoding=\"utf-8\").splitlines(), 1):\n        if not line.strip():\n            continue\n        row = json.loads(line)\n        identifier = str(row[\"id\"])\n        if identifier in predictions:\n            raise SystemExit(f\"Duplicate prediction at {path}:{line_number}\")\n        predictions[identifier] = row\n    return predictions\n\n\ndef main() -> None:\n    parser = argparse.ArgumentParser()\n    parser.add_argument(\"--manifest\", type=Path, required=True)\n    parser.add_argument(\"--predictions\", type=Path, required=True)\n    parser.add_argument(\"--output\", type=Path, required=True)\n    arguments = parser.parse_args()\n    manifest_bytes = arguments.manifest.read_bytes()\n    manifest = json.loads(manifest_bytes)\n    report = {\n        \"schemaVersion\": 1,\n        \"benchmarkID\": manifest[\"benchmarkID\"],\n        \"manifestFileSHA256\": hashlib.sha256(manifest_bytes).hexdigest(),\n        **score_items(list(manifest[\"items\"]), load_predictions(arguments.predictions)),\n    }\n    arguments.output.parent.mkdir(parents=True, exist_ok=True)\n    arguments.output.write_text(\n        json.dumps(report, indent=2, sort_keys=True) + \"\\n\",\n        encoding=\"utf-8\",\n    )\n    print(json.dumps(report, indent=2, sort_keys=True))\n\n\nif __name__ == \"__main__\":\n    main()\n"
}''')
SOURCE_ROOT = Path("/content/voxol-sources")
for relative_path, source in EMBEDDED_SOURCES.items():
    destination = SOURCE_ROOT / relative_path
    destination.parent.mkdir(parents=True, exist_ok=True)
    destination.write_text(source, encoding="utf-8")

print(f"VoxoL sources ready: {SOURCE_ROOT}")


In [ ]:
import subprocess
import sys
from collections import deque

def run_python(relative_script, *arguments):
    command = [sys.executable, str(SOURCE_ROOT / relative_script), *map(str, arguments)]
    print("+", " ".join(command), flush=True)
    log_root = DRIVE_ROOT / "logs"
    log_root.mkdir(parents=True, exist_ok=True)
    log_path = log_root / f"{relative_script.replace('/', '-')}.log"
    tail = deque(maxlen=80)
    with log_path.open("w", encoding="utf-8") as log:
        process = subprocess.Popen(
            command,
            stdout=subprocess.PIPE,
            stderr=subprocess.STDOUT,
            text=True,
            bufsize=1,
        )
        assert process.stdout is not None
        for line in process.stdout:
            print(line, end="")
            log.write(line)
            log.flush()
            tail.append(line)
        return_code = process.wait()
    if return_code != 0:
        diagnostic = "".join(tail).strip() or "The child process produced no output."
        raise RuntimeError(
            f"{relative_script} failed with exit code {return_code}.\n"
            f"Last output:\n{diagnostic}\n"
            f"Full log: {log_path}"
        )

dataset_cache = DRIVE_ROOT / "dataset-cache"
training_data = SCRATCH_ROOT / "data" / "parakeet-fleurs-fr-en"
fleurs_test = SCRATCH_ROOT / "benchmarks" / "fleurs-test"
mediaspeech_test = SCRATCH_ROOT / "benchmarks" / "mediaspeech-fr"
print("Dataset preparation version: 2026-07-27-fleurs-tsv-v2")

run_python(
    "Scripts/prepare-parakeet-fleurs-finetune.py",
    "--cache-root", dataset_cache / "fleurs-training",
    "--output-root", training_data,
)
run_python(
    "Scripts/prepare-fleurs-test-benchmark.py",
    "--cache-root", dataset_cache / "fleurs-test",
    "--output-root", fleurs_test,
)
run_python(
    "Scripts/prepare-mediaspeech-fr-benchmark.py",
    "--cache-root", dataset_cache / "mediaspeech",
    "--output-root", mediaspeech_test,
)
for benchmark_root in (fleurs_test, mediaspeech_test):
    run_python(
        "Tools/training/freeze_asr_manifest.py",
        "--input", benchmark_root / "manifest-unfrozen.json",
        "--output", benchmark_root / "manifest-frozen.json",
        "--timestamp", "2026-07-26T00:00:00Z",
    )

print("Training and locked evaluation datasets are ready.")


In [ ]:
import json
import shutil
import subprocess
import sys

local_experiment_root = SCRATCH_ROOT / "experiments"
durable_candidate_root = DRIVE_ROOT / "candidates"
result_root = DRIVE_ROOT / "results"
local_experiment_root.mkdir(parents=True, exist_ok=True)
durable_candidate_root.mkdir(parents=True, exist_ok=True)
result_root.mkdir(parents=True, exist_ok=True)
completion_path = result_root / "training-complete.json"
print("Training checkpoint version: 2026-07-28-trainable-delta-v1")

def existing_candidate():
    if not completion_path.is_file():
        return None
    completion = json.loads(completion_path.read_text(encoding="utf-8"))
    candidate = Path(completion["candidate"])
    return candidate if candidate.is_file() else None

def stream_process(command, log_path):
    print("+", " ".join(map(str, command)), flush=True)
    with log_path.open("w", encoding="utf-8") as log:
        process = subprocess.Popen(
            list(map(str, command)),
            stdout=subprocess.PIPE,
            stderr=subprocess.STDOUT,
            text=True,
            bufsize=1,
        )
        assert process.stdout is not None
        for line in process.stdout:
            print(line, end="")
            log.write(line)
            log.flush()
        return process.wait()

def persist_candidate(source, destination):
    partial_destination = destination.with_suffix(destination.suffix + ".partial")
    with (
        source.open("rb") as source_file,
        partial_destination.open("wb") as destination_file,
    ):
        shutil.copyfileobj(source_file, destination_file, length=8 * 1024 * 1024)
    if partial_destination.stat().st_size != source.stat().st_size:
        raise RuntimeError("The candidate copy to Google Drive is incomplete.")
    partial_destination.replace(destination)
    return destination

candidate = existing_candidate()
if candidate is None:
    fallback = {
        **PROFILE,
        "name": f"{PROFILE['name']}-oom-fallback",
        "batch_size": 1,
        "validation_batch_size": 1,
        "accumulate_grad_batches": 16,
        "max_duration": min(10, PROFILE["max_duration"]),
        "train_top_encoder_layers": min(4, PROFILE["train_top_encoder_layers"]),
    }
    for attempt_number, attempt in enumerate((PROFILE, fallback), 1):
        attempt_experiment_root = (
            local_experiment_root / f"attempt-{attempt_number}"
        )
        command = [
            sys.executable,
            SOURCE_ROOT / "Tools/training/run_voxol_nemo_finetune.py",
            "--train-manifest", training_data / "train.jsonl",
            "--validation-manifest", training_data / "validation.jsonl",
            "--experiment-root", attempt_experiment_root,
            "--precision", attempt["precision"],
            "--batch-size", attempt["batch_size"],
            "--validation-batch-size", attempt["validation_batch_size"],
            "--accumulate-grad-batches", attempt["accumulate_grad_batches"],
            "--max-duration", attempt["max_duration"],
            "--train-top-encoder-layers", attempt["train_top_encoder_layers"],
            "--max-epochs", 5,
            "--learning-rate", "2e-5",
            "--minimum-learning-rate", "2e-6",
            "--warmup-steps", 100,
            "--num-workers", 2,
        ]
        log_path = result_root / f"training-attempt-{attempt_number}.log"
        return_code = stream_process(command, log_path)
        if return_code == 0:
            candidates = sorted(
                attempt_experiment_root.rglob("*.delta.pt"),
                key=lambda path: path.stat().st_mtime,
            )
            if not candidates:
                raise RuntimeError(
                    "Training completed but produced no trainable delta."
                )
            local_candidate = candidates[-1]
            durable_candidate = (
                durable_candidate_root / f"{attempt['name']}.delta.pt"
            )
            candidate = persist_candidate(local_candidate, durable_candidate)
            completion_path.write_text(
                json.dumps(
                    {
                        "candidate": str(candidate),
                        "profile": attempt,
                        "log": str(log_path),
                    },
                    indent=2,
                    sort_keys=True,
                )
                + "\n",
                encoding="utf-8",
            )
            break
        log_text = log_path.read_text(encoding="utf-8").lower()
        resource_failure = (
            return_code in (-9, 137)
            or "out of memory" in log_text
            or "cuda error: out of memory" in log_text
        )
        if not resource_failure or attempt_number == 2:
            raise RuntimeError(
                f"Training failed with exit code {return_code}; inspect {log_path}"
            )
        print(
            "The training process was killed while exhausting a resource. "
            "Retrying automatically with the safe fallback."
        )
else:
    print(f"Using completed candidate: {candidate}")

print(f"Candidate ready: {candidate}")


In [ ]:
import subprocess
import sys

def evaluated_predictions(label, benchmark_label, benchmark_root, model_arguments):
    output_root = result_root / label
    output_root.mkdir(parents=True, exist_ok=True)
    predictions = output_root / f"{benchmark_label}-predictions.jsonl"
    command = [
        sys.executable,
        SOURCE_ROOT / "Tools/training/run_nemo_asr_benchmark.py",
        *model_arguments,
        "--manifest", benchmark_root / "manifest-frozen.json",
        "--audio-root", benchmark_root / "audio",
        "--output", predictions,
        "--batch-size", PROFILE["evaluation_batch_size"],
        "--resume",
    ]
    print("+", " ".join(map(str, command)), flush=True)
    subprocess.run(list(map(str, command)), check=True)
    report = output_root / f"{benchmark_label}-report.json"
    score_command = [
        sys.executable,
        SOURCE_ROOT / "Tools/training/score_asr_predictions.py",
        "--manifest", benchmark_root / "manifest-frozen.json",
        "--predictions", predictions,
        "--output", report,
    ]
    print("+", " ".join(map(str, score_command)), flush=True)
    subprocess.run(list(map(str, score_command)), check=True)
    return report

reports = {}
for label, model_arguments in (
    ("baseline", ["--pretrained-name", "nvidia/parakeet-tdt-0.6b-v3"]),
    ("candidate", ["--delta", candidate]),
):
    reports[(label, "fleurs")] = evaluated_predictions(
        label,
        "fleurs-fr-en-test",
        fleurs_test,
        model_arguments,
    )
    reports[(label, "mediaspeech")] = evaluated_predictions(
        label,
        "mediaspeech-fr",
        mediaspeech_test,
        model_arguments,
    )

print("Baseline and candidate evaluations are complete.")


In [ ]:
import json

def loaded(label, benchmark):
    return json.loads(reports[(label, benchmark)].read_text(encoding="utf-8"))

baseline_fleurs = loaded("baseline", "fleurs")
candidate_fleurs = loaded("candidate", "fleurs")
baseline_media = loaded("baseline", "mediaspeech")
candidate_media = loaded("candidate", "mediaspeech")

checks = {
    "fleursOverallRegressionAtMost0.5Point": (
        candidate_fleurs["microWER"] <= baseline_fleurs["microWER"] + 0.005
    ),
    "fleursFrenchRegressionAtMost0.5Point": (
        candidate_fleurs["byLanguage"]["french"]["microWER"]
        <= baseline_fleurs["byLanguage"]["french"]["microWER"] + 0.005
    ),
    "fleursEnglishRegressionAtMost0.5Point": (
        candidate_fleurs["byLanguage"]["english"]["microWER"]
        <= baseline_fleurs["byLanguage"]["english"]["microWER"] + 0.005
    ),
    "mediaSpeechImprovesAtLeast10PercentRelative": (
        candidate_media["microWER"] <= baseline_media["microWER"] * 0.90
    ),
    "mediaSpeechEmptyOutputsDecrease": (
        candidate_media["emptyOutputCount"] < baseline_media["emptyOutputCount"]
    ),
}
decision = {
    "sourceGatePassed": all(checks.values()),
    "checks": checks,
    "candidate": str(candidate),
    "baseline": {
        "fleurs": baseline_fleurs,
        "mediaspeech": baseline_media,
    },
    "candidateResults": {
        "fleurs": candidate_fleurs,
        "mediaspeech": candidate_media,
    },
    "nextStep": (
        "Export int8 and int4 Core ML candidates, then run the Mac parity/latency gate."
        if all(checks.values())
        else "Reject this candidate; keep the production model and add independent media-domain training data."
    ),
}
decision_path = result_root / "source-gate.json"
decision_path.write_text(
    json.dumps(decision, indent=2, sort_keys=True) + "\n",
    encoding="utf-8",
)
print(json.dumps(decision, indent=2, sort_keys=True))
print(f"\nSaved decision: {decision_path}")


## Finished

The durable outputs are in `My Drive/VoxoL-Parakeet/results/`. `source-gate.json` contains the
decision and every score; the trainable delta is under `candidates/`. If the gate is false, do not
convert or ship the candidate. If it is true, give Codex the delta and `source-gate.json`; the next
step reconstructs a `.nemo` artifact in a fresh low-memory process before the Mac int8/int4 gate.
